# 💬 Talky v3.0 — Groq Free Tier

### ¿Por qué Groq y no HuggingFace Inference API?

| Problema HF Inference API | Solución Groq |
|---|---|
| 503 — modelo cargando en frío | Siempre en línea, latencia < 1s |
| 403 — permisos por modelo (Llama, etc) | Sin restricciones de acceso |
| Router inestable / proveedor cambia | API REST directa, sin intermediarios |
| Rate limits agresivos sin aviso | Rate limits documentados y generosos |
| Respuestas no-chat (solo text gen) | OpenAI-compatible chat completions |
| Fallos silenciosos difíciles de debuggear | Errores HTTP estándar claros |

**Groq free tier:** 14,400 req/día · 6,000 tokens/min · sin tarjeta de crédito

**Token gratuito:** https://console.groq.com → Sign up → API Keys → Create

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELDA 1 — Instalación
# ═══════════════════════════════════════════════════════════
!pip install -q groq gradio pydantic
print('✅ Dependencias instaladas.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.7 MB/s eta 0:00:00
✅ Dependencias instaladas.


In [ ]:
import os
from getpass import getpass

groq_token = getpass('🔑 Ingresa tu Groq API Key (gsk_...): ').strip()
os.environ['GROQ_API_KEY'] = groq_token

from groq import Groq
try:
    _test_client = Groq(api_key=groq_token)
    _test = _test_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': 'di solo: OK'}],
        max_tokens=5,
    )
    print(f'✅ Groq conectado. Respuesta: {_test.choices[0].message.content}')
except Exception as e:
    print(f'❌ Error de conexión: {e}')
    print('   Verifica tu key en https://console.groq.com/keys')

🔑 Ingresa tu Groq API Key (gsk_...): ··········
✅ Groq conectado. Respuesta: OK! Come posso


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELDA 3 — llm_engine: cliente Groq robusto
# ═══════════════════════════════════════════════════════════
import json
import logging
import os
import re
import time
from typing import Any, Dict, List, Optional

from groq import Groq, APIConnectionError, APIStatusError, RateLimitError

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
log = logging.getLogger('talky')

# ─── Configuración ───────────────────────────────────────
# Modelos Groq disponibles en free tier (todos estables, sin restricciones):
# - llama3-8b-8192      → rápido, muy bueno para español
# - llama3-70b-8192     → mejor calidad, algo más lento
# - mixtral-8x7b-32768  → ventana de contexto grande
# - gemma2-9b-it        → Google Gemma 2, excelente en español

MODEL = os.getenv('TALKY_MODEL', 'llama-3.3-70b-versatile')
TEMPERATURE = float(os.getenv('TALKY_TEMP', '0.5'))
MAX_TOKENS  = int(os.getenv('TALKY_MAX_TOKENS', '450'))
MAX_RETRIES = 3
RETRY_DELAY = 2.0   # segundos, se multiplica por intento
MAX_INPUT_CHARS = 1200
MAX_HISTORY_MSGS = 10  # 5 pares user/assistant

# ─── Protección prompt injection ─────────────────────────
_INJECTION_RE = re.compile(
    r'(ignore\s+(previous|all)\s+instructions?'
    r'|system\s*:\s*'
    r'|<\s*/?system\s*>'
    r'|forget\s+your\s+instructions?'
    r'|act\s+as\s+(?:an?\s+)?(?:unfiltered|evil|jailbreak|dan)'
    r'|\[INST\]|\[\/INST\]|</s>|<s>|<<SYS>>|<</SYS>>)',
    re.IGNORECASE | re.DOTALL,
)

def sanitize(text: str) -> str:
    """Trunca, elimina injection patterns y limpia espacios."""
    if not isinstance(text, str):
        return ''
    text = text.strip()[:MAX_INPUT_CHARS]
    return _INJECTION_RE.sub('[bloqueado]', text)

# ─── Cliente singleton ────────────────────────────────────
_groq_client: Optional[Groq] = None

def get_client() -> Groq:
    global _groq_client
    if _groq_client is None:
        key = os.environ.get('GROQ_API_KEY', '')
        if not key:
            raise RuntimeError('GROQ_API_KEY no encontrada en os.environ.')
        _groq_client = Groq(api_key=key)
        log.info('Cliente Groq creado. Modelo: %s', MODEL)
    return _groq_client

# ─── Trim de historial ────────────────────────────────────
def trim_history(messages: List[Dict]) -> List[Dict]:
    """Separa mensajes de sistema y mantiene solo los últimos N de conversación."""
    system_msgs = [m for m in messages if m.get('role') == 'system']
    conv_msgs   = [m for m in messages if m.get('role') != 'system']
    return system_msgs + conv_msgs[-MAX_HISTORY_MSGS:]

# ─── Parser JSON robusto ──────────────────────────────────
def parse_llm_json(raw: str) -> Dict[str, Any]:
    """
    Extrae JSON de la respuesta del LLM.
    Tolerante a: markdown fences, texto antes/después, JSON malformado.
    Nunca lanza excepción — siempre devuelve un dict usable.
    """
    fallback = {
        'reply': 'Te escucho. ¿Qué es lo que más te está pesando hoy?',
        'profile_update': {},
        'add_disclaimer': False,
    }
    if not raw or not raw.strip():
        log.warning('parse_llm_json: raw vacío, usando fallback.')
        return fallback

    s = raw.strip()
    # Quitar markdown fences
    s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.MULTILINE)
    s = re.sub(r'\s*```$',          '', s, flags=re.MULTILINE).strip()
    # Buscar el objeto JSON si hay texto antes
    if not s.startswith('{'):
        m = re.search(r'\{.*\}', s, re.DOTALL)
        if m:
            s = m.group(0).strip()
        else:
            # No hay JSON — usar el texto completo como reply
            log.warning('parse_llm_json: sin JSON, usando texto plano como reply.')
            return {**fallback, 'reply': raw.strip()}

    try:
        data = json.loads(s)
    except json.JSONDecodeError as e:
        log.warning('parse_llm_json: JSONDecodeError (%s). Usando texto como reply.', e)
        return {**fallback, 'reply': raw.strip()}

    # Normalizar estructura
    if not isinstance(data.get('reply'), str) or not data['reply'].strip():
        data['reply'] = fallback['reply']
    data.setdefault('profile_update', {})
    data.setdefault('add_disclaimer', False)
    return data

# ─── Llamada LLM con reintentos ───────────────────────────
def call_llm(messages: List[Dict], model: str = MODEL) -> str:
    """
    Llama a la API de Groq con reintentos automáticos.
    Lanza excepción si todos los reintentos fallan.
    """
    client = get_client()

    # Sanitizar mensajes de usuario
    safe = []
    for m in messages:
        msg = dict(m)
        if msg.get('role') == 'user':
            msg['content'] = sanitize(msg.get('content', ''))
        safe.append(msg)
    safe = trim_history(safe)

    last_err: Optional[Exception] = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log.info('LLM intento %d/%d — modelo %s', attempt, MAX_RETRIES, model)
            resp = client.chat.completions.create(
                model=model,
                messages=safe,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            content = (resp.choices[0].message.content or '').strip()
            log.info('LLM OK — %d chars', len(content))
            return content

        except RateLimitError as e:
            wait = RETRY_DELAY * attempt * 2  # espera más en rate limit
            log.warning('RateLimitError. Esperando %.1fs antes de reintento.', wait)
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(wait)

        except APIConnectionError as e:
            log.error('APIConnectionError: %s', e)
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)

        except APIStatusError as e:
            log.error('APIStatusError %d: %s', e.status_code, e.message)
            last_err = e
            # No reintentar en errores 4xx (auth, bad request)
            if e.status_code in (400, 401, 403, 404):
                break
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)

        except Exception as e:
            log.error('Error inesperado intento %d: %s', attempt, e)
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

    raise ConnectionError(f'Groq falló después de {MAX_RETRIES} intentos. Último error: {last_err}')


def call_llm_safe(messages: List[Dict], model: str = MODEL) -> str:
    """
    Versión segura de call_llm: NUNCA lanza excepción.
    En caso de error devuelve JSON de fallback con el error en consola.
    El frontend siempre recibe texto usable.
    """
    try:
        return call_llm(messages, model=model)
    except ConnectionError as e:
        log.error('call_llm_safe — ConnectionError: %s', e)
        return json.dumps({
            'reply': (
                '⚠️ Hubo un problema de conexión con el servidor. '
                'Por favor espera unos segundos e intenta de nuevo.'
            ),
            'profile_update': {},
            'add_disclaimer': False,
        }, ensure_ascii=False)
    except Exception as e:
        log.error('call_llm_safe — Error inesperado: %s', e)
        return json.dumps({
            'reply': 'Ocurrió un error inesperado. Inténtalo nuevamente.',
            'profile_update': {},
            'add_disclaimer': False,
        }, ensure_ascii=False)


print(f'✅ llm_engine cargado. Modelo: {MODEL}')

✅ llm_engine cargado. Modelo: llama-3.3-70b-versatile


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELDA 4 — crisis_detection: 3 niveles de riesgo
# ═══════════════════════════════════════════════════════════
import random
import re
from dataclasses import dataclass
from enum import Enum
from typing import List, Optional

class RiskLevel(str, Enum):
    NONE     = 'none'
    LOW      = 'low'
    MODERATE = 'moderate'
    HIGH     = 'high'

@dataclass(frozen=True)
class CrisisResult:
    level:              RiskLevel
    matched_pattern:    Optional[str]
    emergency_response: Optional[str]
    notes:              List[str]

    @property
    def is_emergency(self) -> bool:
        return self.level == RiskLevel.HIGH

_HIGH_P = [
    r'\b(suicid[oa]?|suicidarme|suicidarse)\b',
    r'\b(matarme|quitarme\s+la\s+vida|acabar\s+con\s+(todo|mi\s+vida))\b',
    r'\b(no\s+quiero\s+(seguir\s+)?vivir|ya\s+no\s+quiero\s+vivir)\b',
    r'\b(me\s+voy\s+a\s+hacer\s+da[ñn]o|hacerme\s+da[ñn]o|autolesion[a-z]*)\b',
    r'\b(cortarme|lastimarme)\b',
    r'\b(tengo\s+un\s+arma|voy\s+a\s+(disparar|matar)|plan\s+para\s+morir)\b',
    r'\b(me\s+quiero\s+morir|quiero\s+(estar\s+)?muerto?)\b',
]
_MODERATE_P = [
    r'\b(no\s+tengo\s+(ganas|motivos)\s+de\s+(vivir|nada))\b',
    r'\b(ser[íi]a\s+mejor\s+si\s+(yo\s+)?no\s+existiera)\b',
    r'\b(ojal[áa]\s+(me\s+muriera|no\s+hubiera\s+nacido|no\s+existiera))\b',
    r'\b(nadie\s+me\s+necesita|soy\s+una\s+carga)\b',
    r'\b(todo\s+ser[íi]a\s+mejor\s+sin\s+m[íi])\b',
    r'\b(me\s+he\s+cortado|me\s+he\s+lastimado)\b',
    r'\b(desesperado?|sin\s+esperanza|no\s+hay\s+salida)\b',
]
_LOW_P = [
    r'\b(me\s+siento\s+(muy\s+)?(solo|sola|vac[íi]o|perdido|perdida))\b',
    r'\b(no\s+puedo\s+m[áa]s|estoy\s+agotado?|al\s+l[íi]mite)\b',
    r'\b(todo\s+me\s+pesa|ya\s+no\s+tengo\s+fuerzas)\b',
    r'\b(violencia|me\s+(golpea|pega|lastima|maltrata))\b',
]

_re_high = re.compile('|'.join(_HIGH_P),     re.IGNORECASE | re.DOTALL)
_re_mod  = re.compile('|'.join(_MODERATE_P), re.IGNORECASE | re.DOTALL)
_re_low  = re.compile('|'.join(_LOW_P),      re.IGNORECASE | re.DOTALL)

EMERGENCY_NUMBER = '911'

_EMERG_MSGS = [
    (
        'Lo que me estás diciendo es muy serio y me importa tu seguridad ahora mismo.\n\n'
        f'🆘 Por favor llama a emergencias ({EMERGENCY_NUMBER}) '
        'o pide a alguien de confianza que esté contigo.\n'
        '• Si puedes, aléjate de cualquier objeto con el que pudieras lastimarte.\n'
        '• Si estás en casa, busca un lugar donde no estés solo/a.\n\n'
        '¿Puedes decirme si ahora mismo estás en un lugar seguro?'
    ),
    (
        'Escucho que estás pasando por algo muy doloroso. Tu vida importa.\n\n'
        f'🆘 Llama ahora a {EMERGENCY_NUMBER} o a una línea de crisis.\n'
        'No tienes que atravesar esto solo/a.\n\n'
        '¿Hay alguien de confianza contigo en este momento?'
    ),
]
_MOD_MSGS = [
    (
        'Lo que describes me preocupa, y quiero que sepas que no estás solo/a. '
        'Esos pensamientos son una señal de cuánto dolor estás cargando.\n\n'
        '¿Tienes a alguien —un amigo, familiar o profesional— con quien puedas hablar hoy?'
    ),
    (
        'Lo que sientes tiene peso real, y es importante que recibas apoyo más allá '
        'de esta conversación. Un profesional de salud mental puede acompañarte '
        'de una manera que yo no puedo.\n\n'
        '¿Has podido hablar con alguien de confianza sobre lo que estás sintiendo?'
    ),
]

def detect_crisis(text: str) -> CrisisResult:
    if not text:
        return CrisisResult(RiskLevel.NONE, None, None, [])
    m = _re_high.search(text)
    if m:
        log.warning('🚨 CRISIS ALTA — patrón: %s', m.group(0)[:50])
        return CrisisResult(
            RiskLevel.HIGH, m.group(0),
            random.choice(_EMERG_MSGS),
            ['Posible riesgo inminente.', f"Patrón: '{m.group(0)[:60]}'"]
        )
    m = _re_mod.search(text)
    if m:
        log.info('⚠️  CRISIS MODERADA — patrón: %s', m.group(0)[:50])
        return CrisisResult(
            RiskLevel.MODERATE, m.group(0),
            random.choice(_MOD_MSGS),
            ['Señales de malestar significativo.', f"Patrón: '{m.group(0)[:60]}'"],
        )
    m = _re_low.search(text)
    if m:
        return CrisisResult(
            RiskLevel.LOW, m.group(0), None,
            ['Malestar emocional presente.']
        )
    return CrisisResult(RiskLevel.NONE, None, None, [])

def crisis_to_profile_update(r: CrisisResult) -> dict:
    if r.level != RiskLevel.NONE:
        return {'risk': {'level': r.level.value, 'notes': r.notes}}
    return {}

print('✅ crisis_detection cargado.')

✅ crisis_detection cargado.


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELDA 5 — profile_manager: Pydantic v2
# ═══════════════════════════════════════════════════════════
import time
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field, field_validator, model_validator

class UserContext(BaseModel):
    preferred_name: Optional[str] = None
    age_range:      Optional[str] = None
    location:       Optional[str] = None

    @field_validator('preferred_name','age_range','location', mode='before')
    @classmethod
    def _clean(cls, v):
        return str(v).strip() or None if v else None

class ConversationSignals(BaseModel):
    main_topics:      List[str] = Field(default_factory=list)
    current_emotions: List[str] = Field(default_factory=list)
    stressors:        List[str] = Field(default_factory=list)
    supports:         List[str] = Field(default_factory=list)
    coping:           List[str] = Field(default_factory=list)
    sleep:            Optional[str] = None
    appetite:         Optional[str] = None

    @field_validator('main_topics','current_emotions','stressors','supports','coping', mode='before')
    @classmethod
    def _clean_list(cls, v):
        return [str(i).strip() for i in v if str(i).strip()] if isinstance(v, list) else []

class RiskInfo(BaseModel):
    level: str = 'unknown'
    notes: List[str] = Field(default_factory=list)

    @field_validator('level', mode='before')
    @classmethod
    def _val_level(cls, v):
        allowed = {'unknown','none','low','moderate','high'}
        s = str(v).strip().lower() if v else 'unknown'
        return s if s in allowed else 'unknown'

    @field_validator('notes', mode='before')
    @classmethod
    def _clean_notes(cls, v):
        return [str(n).strip() for n in v if str(n).strip()] if isinstance(v, list) else []

class Goals(BaseModel):
    what_user_wants: Optional[str] = None
    next_step:       Optional[str] = None

class TalkyProfile(BaseModel):
    version:              str = 'groq-3.0'
    updated_at:           Optional[str] = None
    user_context:         UserContext         = Field(default_factory=UserContext)
    conversation_signals: ConversationSignals = Field(default_factory=ConversationSignals)
    risk:                 RiskInfo            = Field(default_factory=RiskInfo)
    goals:                Goals               = Field(default_factory=Goals)

    @model_validator(mode='after')
    def _stamp(self):
        self.updated_at = time.strftime('%Y-%m-%dT%H:%M:%S')
        return self

def _deep_merge(base: dict, upd: dict) -> dict:
    result = dict(base)
    for k, v in upd.items():
        if k not in result:
            continue
        ov = result[k]
        if isinstance(ov, dict) and isinstance(v, dict):
            result[k] = _deep_merge(ov, v)
        elif isinstance(ov, list) and isinstance(v, list):
            seen = {str(x).lower() for x in ov}
            for item in v:
                s = str(item).strip()
                if s and s.lower() not in seen:
                    ov.append(s); seen.add(s.lower())
            result[k] = ov
        else:
            if v is None: continue
            if isinstance(v, str) and not v.strip(): continue
            result[k] = v
    return result

class ProfileManager:
    def __init__(self):
        self._p = TalkyProfile()

    def reset(self):
        self._p = TalkyProfile()

    def merge(self, upd: dict):
        if not isinstance(upd, dict) or not upd:
            return
        try:
            self._p = TalkyProfile(**_deep_merge(self._p.model_dump(), upd))
        except Exception as e:
            log.warning('Profile merge error: %s', e)

    def snapshot(self) -> dict:
        return self._p.model_dump()

    def to_text(self) -> str:
        p = self._p
        icons = {'none':'🟢','unknown':'⚪','low':'🟡','moderate':'🟠','high':'🔴'}
        lines = [
            '╔══════════════════════════════════╗',
            '║  PERFIL PRELIMINAR (no clínico)  ║',
            '╚══════════════════════════════════╝', ''
        ]
        uc = p.user_context
        if any([uc.preferred_name, uc.age_range, uc.location]):
            lines.append('👤 CONTEXTO')
            if uc.preferred_name: lines.append(f'  • Nombre: {uc.preferred_name}')
            if uc.age_range:      lines.append(f'  • Edad aprox: {uc.age_range}')
            if uc.location:       lines.append(f'  • Ubicación: {uc.location}')
            lines.append('')
        cs = p.conversation_signals
        lines.append('💬 SEÑALES')
        if cs.main_topics:      lines.append(f'  • Temas: {", ".join(cs.main_topics)}')
        if cs.current_emotions: lines.append(f'  • Emociones: {", ".join(cs.current_emotions)}')
        if cs.stressors:        lines.append(f'  • Estresores: {", ".join(cs.stressors)}')
        if cs.supports:         lines.append(f'  • Apoyo: {", ".join(cs.supports)}')
        if cs.coping:           lines.append(f'  • Afrontamiento: {", ".join(cs.coping)}')
        lines.append('')
        r = p.risk
        lines.append(f'⚠️  RIESGO: {icons.get(r.level, "⚪")} {r.level.upper()}')
        for n in r.notes: lines.append(f'  • {n}')
        lines += ['', f'🕐 {p.updated_at}', '',
                  '⚕️  Perfil orientativo. No es diagnóstico clínico.']
        return '\n'.join(lines)

print('✅ profile_manager cargado.')

✅ profile_manager cargado.


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELDA 6 — emotional_logic: prompts, openers, anti-repetición
# ═══════════════════════════════════════════════════════════
import random
import re
from collections import deque
from typing import Deque, List, Optional

SYSTEM_PROMPT = """
Eres Talky, un asistente de apoyo emocional y acompañamiento conversacional previo
a una sesión con un profesional de salud mental. Respondes siempre en español.

TU FUNCIÓN:
- Si recibes un "Hola", preséntate de forma cercana y natural.
- Compórtate como un psicólogo profesional en tono, no en intervención clínica.
- Brindar contención emocional genuina y cálida.
- Facilitar que la persona se exprese con libertad.
- Detectar emociones de manera implícita sin necesidad de interrogarlas.
- Solo hacer preguntas cuando realmente ayuden a profundizar o clarificar algo importante.
- Si das consejos (máximo 3 y solo si te los piden), que sean breves y generales.
- Si ves un cambio drástico de tema consulta el por qué y trata de asociarlo a lo psicológico.

COMPORTAMIENTO CONVERSACIONAL:
- No conviertas la conversación en una entrevista.
- No hagas preguntas por obligación.
- Si la persona solo quiere desahogarse, acompaña sin interrogar.
- Puedes responder con reflexión y validación sin cerrar siempre con una pregunta.
- Haz preguntas solo cuando aporten claridad o cuidado.

RESTRICCIONES:
- NO diagnostiques trastornos.
- NO uses etiquetas clínicas.
- NO hagas terapia.
- NO uses frases de coaching o superación personal.
- Máximo una pregunta por turno si decides hacerla.
- Máximo 3 párrafos por respuesta.

ESTILO:
- Lenguaje humano, cálido y específico.
- Valida primero lo que siente la persona.
- Evita sonar estructurado o robótico.
- Que parezca una conversación real, no un formulario.

ÉTICA:
Si detectas riesgo, prioriza seguridad y sugiere apoyo profesional de manera cuidadosa.
""".strip()

RESPONSE_FORMAT = """
Responde ÚNICAMENTE con un objeto JSON válido. Sin texto antes ni después. Sin markdown.

{
  "reply": "Tu respuesta empática en español (1-3 párrafos)",
  "profile_update": {
    "user_context": {"preferred_name": null, "age_range": null, "location": null},
    "conversation_signals": {
      "main_topics": [],
      "current_emotions": [],
      "stressors": [],
      "supports": [],
      "coping": []
    },
    "risk": {"level": "none", "notes": []},
    "goals": {"what_user_wants": null, "next_step": null}
  },
  "add_disclaimer": false
}

REGLAS reply:
- No inicies SOLO con "Lo siento mucho", "Entiendo" o "Claro que sí".
- No uses "¿Parece que...?".
- No repitas frases del turno anterior.
- Máximo UNA pregunta.
- Sin coaching. Sin metas. En español.

REGLAS profile_update:
- Solo incluye lo mencionado explícitamente por el usuario.
- No inventes. No uses etiquetas diagnósticas.
- Omite campos sin información nueva.

JSON ESTRICTO. Cero texto adicional.
""".strip()

DISCLAIMERS = [
    'Antes de continuar, quiero aclararte que puedo acompañarte, pero esto no reemplaza la atención de un profesional de la salud mental.',
    'Este espacio es de apoyo emocional orientativo; no sustituye un proceso terapéutico profesional.',
    'Puedo escucharte y ayudarte a ordenar ideas, aunque no realizo diagnósticos clínicos.',
    'Mi rol es acompañarte y ofrecerte reflexión, no sustituir la atención de un especialista.',
]

OPENERS = [
    'Te escucho.',
    'Suena a que esto ha sido pesado.',
    'Lo que describes refleja un momento difícil.',
    'Tiene sentido que esto te afecte.',
    'Se nota que esto te ha movido emocionalmente.',
    'Lo que sientes tiene mucho peso.',
    'Veo que esto te está ocupando bastante.',
    'Por lo que me cuentas, has estado cargando con mucho.',
]

_BAD_STARTS = (
    'gracias por contármelo', 'gracias por compartirlo',
    'lo siento mucho', 'lamento mucho', 'entiendo perfectamente',
    'claro que sí', 'por supuesto',
    'por lo que describes', 'parece que sientes',
)

def pick_opener(used: deque) -> str:
    for op in OPENERS:
        if op not in used:
            used.append(op)
            return op
    used.clear()
    used.append(OPENERS[0])
    return OPENERS[0]

def fix_opener(reply: str, used: deque) -> str:
    if any(reply.strip().lower().startswith(b) for b in _BAD_STARTS):
        opener = pick_opener(used)
        if '.' in reply[:120]:
            reply = reply.split('.', 1)[1].strip()
        return f'{opener} {reply}'.strip()
    return reply

def bigram_similarity(a: str, b: str) -> float:
    def bigrams(s):
        w = re.sub(r'\s+', ' ', (s or '').lower().strip()).split()
        return set(zip(w, w[1:]))
    ba, bb = bigrams(a), bigrams(b)
    if not ba or not bb: return 0.0
    return len(ba & bb) / len(ba | bb)

def too_similar(a: str, b: str, thr: float = 0.65) -> bool:
    return bigram_similarity(a, b) >= thr

def build_messages(user_text: str, memory: list, profile_snap: dict,
                   extra: Optional[str] = None) -> list:
    hint = json.dumps({'perfil': {
        k: profile_snap.get(k, {})
        for k in ('user_context','conversation_signals','risk','goals')
    }}, ensure_ascii=False, indent=2)

    instructions = RESPONSE_FORMAT
    if extra:
        instructions += f'\n\nINSTRUCCIÓN ADICIONAL: {extra}'

    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'system', 'content': f'Contexto acumulado:\n{hint}'},
        *memory,
        {'role': 'system', 'content': instructions},
        {'role': 'user',   'content': user_text.strip()},
    ]

def apply_final_polish(reply: str, add_disc: bool, disc_shown: bool,
                       used_openers: deque) -> tuple:
    reply = fix_opener(reply, used_openers)
    if not disc_shown and (add_disc or True):  # siempre en primer turno
        reply = f'_{random.choice(DISCLAIMERS)}_\n\n{reply}'
        return reply, True
    return reply, disc_shown

print('✅ emotional_logic cargado.')

✅ emotional_logic cargado.


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELDA 7 — App completa: orquestación + Gradio
# ═══════════════════════════════════════════════════════════
import gradio as gr
from collections import deque
from dataclasses import dataclass, field
from typing import List, Tuple

MAX_MEM = 10

@dataclass
class Session:
    memory:       List[dict]     = field(default_factory=list)
    pm:           ProfileManager = field(default_factory=ProfileManager)
    disc_shown:   bool           = False
    used_openers: deque          = field(default_factory=lambda: deque(maxlen=4))
    last_reply:   str            = ''
    turn:         int            = 0

    def reset(self):
        self.memory.clear()
        self.pm.reset()
        self.disc_shown = False
        self.used_openers.clear()
        self.last_reply = ''
        self.turn = 0


def _trim(mem):
    return mem[-MAX_MEM:] if len(mem) > MAX_MEM else mem


def talky_turn(user_input, sess):
    user_input = sanitize(user_input)
    if not user_input:
        return 'Estoy aquí. ¿Qué tienes en mente ahora mismo?', sess

    crisis = detect_crisis(user_input)

    if crisis.level == RiskLevel.HIGH:
        sess.pm.merge(crisis_to_profile_update(crisis))
        sess.memory.append({'role': 'user',      'content': user_input})
        sess.memory.append({'role': 'assistant', 'content': crisis.emergency_response})
        sess.turn += 1
        return crisis.emergency_response, sess

    crisis_prefix = crisis.emergency_response if crisis.level == RiskLevel.MODERATE else None

    msgs = build_messages(user_input, _trim(sess.memory), sess.pm.snapshot())
    raw  = call_llm_safe(msgs)
    data = parse_llm_json(raw)
    reply = data.get('reply', '').strip() or 'Te escucho. ¿Qué es lo que más te está pesando hoy?'

    if too_similar(reply, sess.last_reply):
        log.info('Respuesta repetida — solicitando variación.')
        msgs2 = build_messages(
            user_input, _trim(sess.memory), sess.pm.snapshot(),
            extra='Usa vocabulario completamente diferente y formula una pregunta distinta.'
        )
        raw2  = call_llm_safe(msgs2)
        data2 = parse_llm_json(raw2)
        r2    = data2.get('reply', '').strip()
        if r2:
            reply, data = r2, data2

    upd = data.get('profile_update', {})
    if isinstance(upd, dict) and upd:
        sess.pm.merge(upd)
    if crisis.level in (RiskLevel.MODERATE, RiskLevel.LOW):
        sess.pm.merge(crisis_to_profile_update(crisis))

    reply, sess.disc_shown = apply_final_polish(
        reply, bool(data.get('add_disclaimer', False)),
        sess.disc_shown, sess.used_openers
    )
    if crisis_prefix:
        reply = f'{crisis_prefix}\n\n---\n\n{reply}'

    sess.memory.append({'role': 'user',      'content': user_input})
    sess.memory.append({'role': 'assistant', 'content': reply})
    sess.memory     = _trim(sess.memory)
    sess.last_reply = reply
    sess.turn      += 1

    return reply, sess


def chat_fn(user_msg, history, sess):
    if not (user_msg or '').strip():
        return history, '', sess
    try:
        reply, sess = talky_turn(user_msg.strip(), sess)
    except Exception as e:
        log.error('chat_fn error: %s', e)
        reply = '⚠️ Ocurrió un problema técnico. Por favor intenta de nuevo.'
    return history + [[user_msg, reply]], '', sess

def finalize_fn(sess):
    return sess.pm.to_text(), sess

def reset_fn():
    return [], '', '', Session()

CSS = """
/* ===== Fondo general oscuro premium ===== */
body {
    background: linear-gradient(135deg, #0f172a, #1e293b);
    min-height: 100vh;
}

/* ===== Contenedor principal ===== */
.gradio-container {
    background: transparent !important;
    color: #e2e8f0 !important;
    font-family: Inter, ui-sans-serif, system-ui;
}

/* ===== Tarjeta central ===== */
#app-wrap {
    max-width: 1100px;
    margin: 60px auto;
    padding: 40px;
    background: #111827;
    border-radius: 30px;
    box-shadow: 0 30px 70px rgba(0,0,0,0.5);
}

/* ===== Chat container ===== */
#chat {
    background: #0b1220 !important;
    border-radius: 25px !important;
    padding: 25px !important;
}

/* ===== Mensajes usuario ===== */
.message.user,
.message.user .content {
    background: linear-gradient(135deg, #14b8a6, #0f766e) !important;
    color: #ffffff !important;
    border-radius: 20px 20px 6px 20px !important;
    padding: 14px 18px !important;
}

/* ===== Mensajes bot ===== */
.message.assistant,
.message.bot,
.message.assistant .content {
    background: #1f2937 !important;
    color: #f1f5f9 !important;
    border-radius: 20px 20px 20px 6px !important;
    padding: 14px 18px !important;
}

/* ===== Input ===== */
textarea {
    background: #1f2937 !important;
    color: #f1f5f9 !important;
    border: 1px solid #334155 !important;
    border-radius: 20px !important;
    padding: 16px !important;
}

textarea:focus {
    border: 1px solid #14b8a6 !important;
    box-shadow: 0 0 0 3px rgba(20,184,166,0.25);
}

/* ===== Botón ===== */
button.primary {
    background: linear-gradient(135deg,#14b8a6,#0f766e) !important;
    color: white !important;
    border-radius: 20px !important;
    border: none !important;
    padding: 12px 22px !important;
    font-weight: 600;
}

button.primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 8px 20px rgba(20,184,166,0.35);
}

/* ===== Sidebar ===== */
#profile {
    background: #1f2937 !important;
    border-radius: 20px !important;
    color: #e2e8f0 !important;
}

/* ===== Ocultar footer ===== */
footer { display: none !important; }
"""

WELCOME = [[None, 'Hola, soy Talky 👋\n\nEstoy aquí para escucharte. Cuéntame, ¿qué está pasando para ti hoy?']]

demo = gr.Blocks(
    title="Talky",
    css=CSS,
    theme=gr.themes.Base()
)
with demo:
    s = gr.State(lambda: Session())

    with gr.Column(elem_id='app-wrap'):
        gr.Markdown("""
            # 💬 Talky
            ### Tu espacio seguro para hablar
            <sub>Apoyo emocional inicial · No clínico · Gratuito</sub>
            """)

        with gr.Row():
            with gr.Column(scale=3, min_width=280):
                end_btn   = gr.Button('✅ Generar perfil',         variant='primary')
                reset_btn = gr.Button('🔄 Reiniciar conversación')
                gr.Markdown('---')
                gr.Markdown('### 📋 Perfil generado')
                with gr.Accordion('Ver perfil', open=True):
                    profile_out = gr.Textbox(
                        lines=22, interactive=False, elem_id='profile',
                        placeholder='Presiona "Generar perfil" al terminar…'
                    )
                gr.Markdown(
                    '---\n'
                    '⚠️ **Aviso:** No reemplaza atención profesional.\n\n'
                    '**Ecuador:** 911 · 1800-SALUD\n'
                    '**España:** 024'
                )

            with gr.Column(scale=7):
                chatbot = gr.Chatbot(
                    label='Conversación con Talky',
                    height=530, elem_id='chat',
                    value=WELCOME, show_copy_button=True
                )
                with gr.Row():
                    msg_in   = gr.Textbox(
                        label='Tu mensaje',
                        placeholder='Cuéntame qué está pasando…',
                        lines=3, scale=8, max_lines=6
                    )
                    send_btn = gr.Button('Enviar ➤', variant='primary', scale=2)
                gr.Examples(
                    [
                        'Me siento ansioso y no sé por qué.',
                        'Se murió mi perrito y me siento muy mal.',
                        'No puedo dormir por pensar demasiado.',
                        'Siento que nadie me entiende.',
                        'Estoy muy estresado con el trabajo.',
                    ],
                    inputs=msg_in, label='Ejemplos'
                )

    send_btn.click(chat_fn, [msg_in, chatbot, s], [chatbot, msg_in, s])
    msg_in.submit( chat_fn, [msg_in, chatbot, s], [chatbot, msg_in, s])
    end_btn.click(   finalize_fn, [s], [profile_out, s])
    reset_btn.click( reset_fn,    [],  [chatbot, msg_in, profile_out, s])


demo.queue()
demo.launch(
    share=True,
    server_name='0.0.0.0',
)
print('\n✅ Talky v3.0 (Groq) en línea.')

/tmp/ipykernel_463/2805323872.py:196: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  demo = gr.Blocks(
/tmp/ipykernel_463/2805323872.py:196: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  demo = gr.Blocks(
/tmp/ipykernel_463/2805323872.py:230: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_463/2805323872.py:230: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykern

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2e3cfcce761ce56ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Talky v3.0 (Groq) en línea.
